In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

class SpatialClusterHead(nn.Module):
    def __init__(self, feature_dim, num_clusters):
        super().__init__()
        self.spatial_mlp = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, num_clusters)
        )
        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_clusters)
        )
        
        # THE FIX: Equalizers
        # This prevents the feature_mlp from bullying the spatial_mlp
        self.spatial_norm = nn.LayerNorm(num_clusters)
        self.feature_norm = nn.LayerNorm(num_clusters)
        
    def forward(self, X_combined, spatial_weight=3.0):
        spatial_logits = self.spatial_mlp(X_combined[:, :2])
        feature_logits = self.feature_mlp(X_combined)
        
        # Normalize both to unit variance before combining
        s = spatial_logits / (spatial_logits.std() + 1e-8)
        f = feature_logits / (feature_logits.std() + 1e-8)
        
        
        return spatial_weight * s + f

class FirstTerm(nn.Module):
    def __init__(self, num_cell_types, num_of_clusters , embedding_dim=8):
        super().__init__()
        self.cell_embedding = nn.Embedding(num_cell_types, embedding_dim)
        # Initialize 10.9M weights (The Dials)
        feature_dim = 18 + embedding_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(feature_dim*2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        nn.init.normal_(self.edge_mlp[2].weight, mean=0.0, std=0.001)
        nn.init.constant_(self.edge_mlp[2].bias, 0.01)


        # 3. THE DUAL-PATH CLUSTER HEAD
        # Replaces the standard Sequential MLP
        self.cluster_head = SpatialClusterHead(
            feature_dim=feature_dim, 
            num_clusters=num_of_clusters
        )



    def forward(self, X, X_cell_ids, num_nodes, p_indices, A_skip_csr, current_k, tau=1.0):
        
        X_cell_ids = X_cell_ids.squeeze()
        cell_features = self.cell_embedding(X_cell_ids)  # Shape: [num_nodes, embedding_dim]
        X_combined = torch.cat([X, cell_features], dim=1)  # Shape: [num_nodes, 18 + embedding_dim]

        src_features = X_combined[p_indices[0]]  # Shape: [num_edges, feature_dim]
        dst_features = X_combined[p_indices[1]]  # Shape: [num_edges, feature_dim]
        edge_inputs = torch.cat([src_features, dst_features], dim=1)  # Shape: [num_edges, feature_dim*2]

        dynamic_p_weights = self.edge_mlp(edge_inputs).squeeze(-1)
        safe_weights = F.softplus(dynamic_p_weights) 


        


        # Enforce P >= 0 and build sparse matrix
        P = torch.sparse_coo_tensor(p_indices, safe_weights, 
                                    (num_nodes, num_nodes)).coalesce()
        
        # Reconstruction: XP
        X_hat = torch.sparse.mm(P, X)
        
        # Loss: ||X - XP||
        error = X - X_hat
        loss1 = torch.mean(error**2)   

        # Pass all node features through the head
        logits = self.cluster_head(X_combined) # Shape: [n, k]

        logits = logits[:, :current_k]
        
        #TERM2
        #C matrix with probability distribution across clusters for each node
        C = F.gumbel_softmax(logits, tau=tau, hard=False) # Shape: [n, k]
        # Inside FirstTerm.forward or the training loop
        # tau_smooth = 0.6  # Higher = more spread out/soft
        # C = F.softmax(logits / tau_smooth, dim=-1)
        # C = F.softmax(logits, dim=-1)  # Ensure positivity for SDDMM

        p_vals = P.values()
        
        # 1. Sum across rows (dim=1) to get the total weight leaving each node
        row_sums = torch.sparse.sum(P, dim=1).to_dense()
        
        # 2. Expand row_sums to match the non-zero values 
        # P.indices()[0] contains the row index for every specific edge
        p_vals_norm = p_vals / (row_sums[P.indices()[0]] + 1e-8)
        
        # Rebuild using the exact same sorted indices
        P_norm = torch.sparse_coo_tensor(P.indices(), p_vals_norm, 
                                         (num_nodes, num_nodes)).coalesce()
        
        # 2. Convert to CSR format (Required for the CUDA SDDMM engine)
        P_csr = P_norm.to_sparse_csr()
        
        # 3. SDDMM Magic! 
        # beta=1.0, alpha=-1.0 calculates exactly: (1.0 * P_csr) - (1.0 * C @ C^T)
        # It ONLY calculates this at the 10.9M non-zero locations!
        diff_csr = torch.sparse.sampled_addmm(P_csr, C, C.t(), beta=1.0, alpha=-1.0)
        
        # 4. Square the differences and sum them
        loss2 = torch.sum(diff_csr.values() ** 2)


        #TERM3
        M = torch.matmul(C.t(), torch.sparse.mm(A_skip_csr, C))  # [k, n] @ [n, n] @ [n, k] -> [k, k]
        # 2. Normalize M into a probability distribution (M_tilde)
        M = torch.clamp(M, min=0)
        M_tilde = M / (M.sum() + 1e-8)
        loss3 = -torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        # 3. Calculate Shannon Entropy: -sum(p * log(p))
        # We only calculate for non-zero entries to avoid log(0)
        # loss3 = torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        

        alpha_1 = 1.0 
        alpha_2 = 1.0    
        alpha_3 = 1.0    
        loss = alpha_1* loss1 + (alpha_2 * loss2) +(alpha_3 * loss3) 



        return  loss, loss1 , loss2, loss3,  C , X_combined


import math
def get_tau(epoch):
    tau_start = 2.0
    tau_mid = 1.35   # Targets ~25% Confidence for the middle flatline
    tau_end = 0.85   # Targets ~40% Confidence for the final floor

    if epoch < 75:
        # Phase 1: Smooth descent to the middle flatline
        progress = epoch / 75.0
        return tau_mid + 0.5 * (tau_start - tau_mid) * (1 + math.cos(math.pi * progress))
        
    elif epoch < 150:
        # Phase 2: THE MIDDLE FLATLINE (Epoch 75 to 150)
        return tau_mid
        
    elif epoch < 200:
        # Phase 3: Smooth descent to the final floor
        progress = (epoch - 150) / 50.0
        return tau_end + 0.5 * (tau_mid - tau_end) * (1 + math.cos(math.pi * progress))
        
    else:
        # Phase 4: THE FINAL FLATLINE (Epoch 200 to 250+)
        return tau_end

In [ ]:
#Graph Neural Network Inference on the compressed graph (X_tilde, A_tilde_skip) 

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GraphConv, to_hetero , global_mean_pool



MASKING_THRESH = 0.20
HIDDEN_DIM = 64

# ==========================================
# 🧠 MULTI-TASK MODEL
# ==========================================
class BaseGNN(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.conv1 = GraphConv((-1, -1), hidden_dim)
        self.conv2 = GraphConv((-1, -1), hidden_dim)
    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        x = F.relu(self.conv2(x, edge_index, edge_weight))
        return x

class HeteroCTS_TripleHead_GNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_params):
        super().__init__()
        self.metadata = (['supernode'], [('supernode', 'physical', 'supernode'), ('supernode', 'timing', 'supernode')])
        self.proj = nn.Linear(input_dim, hidden_dim)
        self.gnn = to_hetero(BaseGNN(hidden_dim), self.metadata, aggr='mean')
        
        # input: pooled_gnn + knobs + n/k ratio
        shared_dim = hidden_dim + num_params + 1

        self.power_head = nn.Sequential(nn.Linear(shared_dim, hidden_dim//2), nn.ReLU(), nn.Linear(hidden_dim//2, 1))
        self.skew_head  = nn.Sequential(nn.Linear(shared_dim, hidden_dim//2), nn.ReLU(), nn.Linear(hidden_dim//2, 1))
        self.wire_head  = nn.Sequential(nn.Linear(shared_dim, hidden_dim//2), nn.ReLU(), nn.Linear(hidden_dim//2, 1))

    def forward(self, x_dict, edge_index_dict, cts_params, complexity_feat, edge_weight_dict=None):
        x = {k: self.proj(v) for k, v in x_dict.items()}
        h_dict = self.gnn(x, edge_index_dict, edge_weight_dict)
        h_pooled = torch.mean(h_dict['supernode'], dim=0, keepdim=True) 
        context = torch.cat([h_pooled, cts_params, complexity_feat], dim=-1)
        
        return {
            'power': self.power_head(context),
            'skew': self.skew_head(context),
            'wire': self.wire_head(context)
        }
 

In [ ]:
import os , random
processed_dir = "processed_graphs"
# List all processed file names
design_files = [f for f in os.listdir(processed_dir) if f.endswith('.pt')]

grouped_designs = {}
for f in design_files:
    # Splits "aes_run_20260306..." to extract just "aes"
    base_name = f.split('_run_')[0] 
    if base_name not in grouped_designs:
        grouped_designs[base_name] = []
    grouped_designs[base_name].append(f)

# 2. Pick exactly 5 from each category
balanced_files = []
for name, files in grouped_designs.items():
    random.shuffle(files)          # Shuffle within the category
    selected = files[:5]           # Take exactly 5 (or less if a category has < 5)
    balanced_files.extend(selected)

# 3. Final shuffle so the epoch isn't ordered by chip type
random.shuffle(balanced_files)
design_files = balanced_files      # Overwrite the main list

print(f"🔥 Fast-Tracking on {len(design_files)} perfectly balanced designs!")
# Quick check to prove it's balanced
print("Balance Check:", {name: len([x for x in design_files if name in x]) for name in grouped_designs.keys()})


In [ ]:
import os
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GraphConv, to_hetero

from helper import load_cts_parameters, get_compressed_graph, relative_masking

# ==========================================
# ⚙️ CONFIGURATION & DEVICE
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
csv_file = "./dataset_with_def/unified_manifest_normalized.csv"
processed_dir = "processed_graphs"


print("❄️ Loading Phase 1 Brain (Frozen Mode)...")
# Note: Ensure FirstTerm class is defined in your helper or script
meta = torch.load("global_metadata.pt", map_location=device)
phase1_model = FirstTerm(num_cell_types=meta['global_max_cell_types'], 
                         num_of_clusters=meta['global_max_k']).to(device)
phase1_model.load_state_dict(torch.load("phase1_clustering_ep250_twohop.pt", map_location=device))
phase1_model.eval()

gnn_model = HeteroCTS_TripleHead_GNN(input_dim=28, hidden_dim=HIDDEN_DIM, num_params=4).to(device)
optimizer = optim.Adam(gnn_model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
criterion = nn.MSELoss()

print("🚀 Initiating Triple-Head Production Run...")

for epoch in range(100):
    gnn_model.train()
    metrics = {'power': 0.0, 'skew': 0.0, 'wire': 0.0, 'total_preds': 0}
    
    # Randomize the file order every epoch for better generalization
    random.shuffle(design_files)

    for filename in design_files:
        placement_id = filename.replace('.pt', '')
        # Direct loading from disk to save RAM
        data = torch.load(os.path.join(processed_dir, filename), map_location=device)
        
        cts_runs = load_cts_parameters(csv_file, placement_id, device)
        if not cts_runs: continue

        # 1. ON-THE-FLY COMPRESSION
        with torch.no_grad():
            _, _, _, _, C, X_comb = phase1_model(
                data['X'].to(device), data['X_cell_ids'].to(device), 
                data['num_nodes'], data['p_indices'].to(device), 
                data['A_skip_csr'].to(device), data['current_k'], tau=0.85
            )
            X_t, A_ts, A_tw = get_compressed_graph(
                X_comb, C, data['A_skip_csr'].to(device), 
                data['A_wire_csr'].to(device), data['raw_areas'].to(device)
            )
            w_idx, w_wt = relative_masking(A_tw, threshold=MASKING_THRESH)
            s_idx, s_wt = relative_masking(A_ts, threshold=MASKING_THRESH)

        # 2. GNN SETUP
        x_dict = {'supernode': X_t}
        e_idx = {('supernode','physical','supernode'): w_idx, ('supernode','timing','supernode'): s_idx}
        e_wt = {('supernode','physical','supernode'): w_wt, ('supernode','timing','supernode'): s_wt}
        complexity_feat = torch.tensor([[data['X'].size(0)/data['current_k']]], device=device)

        # 3. TASK LOOP (Forward & Metrics)
        batch_loss = 0
        for run in cts_runs:
            preds = gnn_model(x_dict, e_idx, run['knobs'].view(1,-1), complexity_feat, e_wt)
            
            t_p, t_s, t_w = run['targets']['power'], run['targets']['skew'], run['targets']['wl']
            
            loss = criterion(preds['power'], t_p.view(1,1)) + \
                   criterion(preds['skew'], t_s.view(1,1)) + \
                   criterion(preds['wire'], t_w.view(1,1))
            
            batch_loss += loss
            metrics['power'] += abs(preds['power'].item() - t_p.item())
            metrics['skew']  += abs(preds['skew'].item() - t_s.item())
            metrics['wire']  += abs(preds['wire'].item() - t_w.item())
            metrics['total_preds'] += 1

        # 4. OPTIMIZE DESIGN BATCH
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()
        
        # Explicitly clean up data objects to keep VRAM/RAM low
        del data, X_comb, X_t, C, w_idx, s_idx
        torch.cuda.empty_cache()

    # Log and Step Scheduler
    p_mae, s_mae, w_mae = metrics['power']/metrics['total_preds'], metrics['skew']/metrics['total_preds'], metrics['wire']/metrics['total_preds']
    scheduler.step(p_mae + s_mae + w_mae)
    
    print(f"Epoch {epoch:2} | Power MAE: {p_mae:.4f} | Skew MAE: {s_mae:.4f} | Wire MAE: {w_mae:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")